In [1]:
# 1. 必要なライブラリのインストール
!pip install ripser persim umap-learn scipy scikit-learn

import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import pairwise_distances
import umap
from ripser import ripser
from persim import plot_diagrams
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

# 2. 実験設定
OUT_DIR = "H6_Smoothing_Ineffectiveness"
os.makedirs(OUT_DIR, exist_ok=True)

trial_scales = [1_000, 10_000, 100_000, 1_000_000, 5_000_000, 10_000_000]

N_PER_GROUP = 100
N_SAMPLES = N_PER_GROUP * 2
results = []

print("=== H6補強検証：統計的平滑化の無効性とTDA事後監査の必然性 ===")

def clr_transform(X, epsilon=1e-10):
    X_eps = X + epsilon
    log_X = np.log(X_eps)
    return log_X - np.mean(log_X, axis=1, keepdims=True)

# 3. メインループ
for i, total_trials in enumerate(trial_scales):
    step = i + 1
    D_features = max(10, total_trials // N_SAMPLES)
    print(f"\n[Step {step}/6] 試行: {total_trials:,} (D={D_features})")

    np.random.seed(42 + step)

    # --- A. 真の異質集団の生成 ---
    X_A = np.random.uniform(0.01, 0.2, size=(N_PER_GROUP, D_features))
    X_B = np.random.uniform(0.7, 0.9, size=(N_PER_GROUP, D_features))
    X_raw = np.vstack([X_A, X_B])

    # --- B. Zipf則（べき乗則）に基づく極端な過分散の注入 ---
    ranks = np.arange(1, D_features + 1)
    base_skew = 10.0 + (step * 5.0)
    zipf_multipliers = 1.0 + base_skew / (ranks ** 0.8) # 裾が重い分布

    X_skewed = X_raw * zipf_multipliers

    # --- C. 実務家がよくやる「統計的調整（平滑化）」の適用 ---
    # 対策1: Log1p 変換 (TF-IDFなどで使われる対数平滑化を模倣)
    X_smoothed = np.log1p(X_skewed)

    # 組成データ（simplex）へ正規化
    X_smoothed_simplex = X_smoothed / np.sum(X_smoothed, axis=1, keepdims=True)

    # --- D. UMAPによる次元圧縮（平滑化してもキメラ化するか？） ---
    dist_cos_smoothed = pairwise_distances(X_smoothed_simplex + 1e-10, metric='cosine')
    reducer = umap.UMAP(n_components=2, metric='precomputed', random_state=42)
    umap_emb = reducer.fit_transform(dist_cos_smoothed)

    # UMAP上での重心間距離（キメラ化の度合い）
    centroid_A = np.mean(umap_emb[:N_PER_GROUP], axis=0)
    centroid_B = np.mean(umap_emb[N_PER_GROUP:], axis=0)
    umap_cluster_distance = np.linalg.norm(centroid_A - centroid_B)

    # --- E. TDAによる事後監査（Aitchison空間） ---
    dist_aitchison = pairwise_distances(clr_transform(X_smoothed_simplex), metric='euclidean')
    ph_res = ripser(dist_aitchison, distance_matrix=True, maxdim=1)['dgms']

    h0_deaths = np.sort(ph_res[0][:, 1][ph_res[0][:, 1] < np.inf])
    chimera_gap = h0_deaths[-1] if len(h0_deaths) > 0 else 1e-5

    results.append({
        'Step': step, 'Dimensions': D_features,
        'UMAP_Visual_Distance': umap_cluster_distance,
        'TDA_Chimera_Gap': chimera_gap
    })

    # --- F. グラフ出力 ---
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(f'Step {step}: Failure of Statistical Smoothing (D={D_features})', fontsize=16)

    # 1. UMAP（平滑化後でもキメラができることを証明）
    axes[0].scatter(umap_emb[:N_PER_GROUP, 0], umap_emb[:N_PER_GROUP, 1], label='Group A', alpha=0.7)
    axes[0].scatter(umap_emb[N_PER_GROUP:, 0], umap_emb[N_PER_GROUP:, 1], label='Group B', alpha=0.7, color='red')
    axes[0].set_title(f'UMAP after Log-Smoothing\nCentroid Distance: {umap_cluster_distance:.3f}')
    axes[0].legend()

    # 2. TDAによる真実の暴露
    plot_diagrams(ph_res, ax=axes[1])
    axes[1].set_title(f'TDA Sanity Check\nChimera Gap: {chimera_gap:.2f}')
    axes[1].axhline(y=chimera_gap, color='gray', linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/smoothing_fail_step_{step}.png")
    plt.close()

# 4. 集計とまとめグラフ
df_results = pd.DataFrame(results)
df_results.to_csv(f"{OUT_DIR}/smoothing_summary.csv", index=False)

fig, ax1 = plt.subplots(figsize=(9, 6))
ax1.plot(trial_scales, df_results['UMAP_Visual_Distance'], marker='o', color='blue', label='UMAP Apparent Distance (Log-Smoothed)')
ax1.set_xlabel('Total Trials (log scale)')
ax1.set_xscale('log')
ax1.set_ylabel('Visual Distance in UMAP', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')

ax2 = ax1.twinx()
ax2.plot(trial_scales, df_results['TDA_Chimera_Gap'], marker='s', color='red', label='High-D TDA Chimera Gap')
ax2.set_ylabel('TDA Chimera Gap', color='red')
ax2.tick_params(axis='y', labelcolor='red')

fig.suptitle('Ineffectiveness of Statistical Smoothing on Heavy-Tailed Data', fontsize=14)
plt.title('Log-transform fails to prevent UMAP chimeras, necessitating TDA audit.', fontsize=10)
fig.tight_layout()
plt.grid(True, which="both", ls="--", alpha=0.5)

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='center right')
plt.savefig(f"{OUT_DIR}/smoothing_ineffectiveness_plot.png")
plt.close()

shutil.make_archive(OUT_DIR, 'zip', OUT_DIR)
print(f"\n実験完了。全データを {OUT_DIR}.zip に圧縮しました。")
files.download(f"{OUT_DIR}.zip")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.1/842.1 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 1.8 MB/s eta 0:00:00
  Created wheel for hopcroftkarp: filename=hopcroftkarp-1.2.5-py2.py3-none-any.whl size=18187 sha256=7c2aae26f11f009ef59573e2c29d994ee9b751695267bf08a0e7b7d47d72a276
  Stored in directory: /root/.cache/pip/wheels/b3/32/77/3c78fadb9f917d5a5546ed0409574012401a83c544ec95999a
Successfully built hopcroftkarp
=== H6補強検証：統計的平滑化の無効性とTDA事後監査の必然性 ===

[Step 1/6] 試行: 1,000 (D=10)

[Step 2/6] 試行: 10,000 (D=50)

[Step 3/6] 試行: 100,000 (D=500)

[Step 4/6] 試行: 1,000,000 (D=5000)

[Step 5/6] 試行: 5,000,000 (D=25000)

[Step 6/6] 試行: 10,000,000 (D=50000)

実験完了。全データを H6_Smoothing_Ineffectiveness.zip に圧縮しました。


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>